In [0]:
"""
id: python_1
template: python
templateVersion: 1.0.0
name: bookings_API
position:
  x: 381.5507760352265
  y: 430.97777434813923
description:
  text: Load data from input if available; otherwise, fetch and load data from an online CSV file.
  hash: 1f791a2b
previewCodeHash: 277dfdeaca4d82d7
previewMode: "1000"
config:
  code: |
    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import pandas as pd
        from io import StringIO
        import requests
        url = 'https://raw.githubusercontent.com/anshlambagit/Airbnb_Snowflake_DBT_Data_Engineer_Project/refs/heads/main/SourceData/bookings.csv'
        response = requests.get(url)
        csv_data = response.content.decode('utf-8')
        df = pd.read_csv(StringIO(csv_data))
        result = spark.createDataFrame(df)
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import pandas as pd
        from io import StringIO
        import requests
        url = 'https://raw.githubusercontent.com/anshlambagit/Airbnb_Snowflake_DBT_Data_Engineer_Project/refs/heads/main/SourceData/bookings.csv'
        response = requests.get(url)
        csv_data = response.content.decode('utf-8')
        df = pd.read_csv(StringIO(csv_data))
        result = spark.createDataFrame(df)

    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["python_1.result"] = out["result"]
if globals().get("ld_display_outputs", False):
    display(ctx["python_1.result"])

In [0]:
"""
id: source_0
template: source
templateVersion: 2.0.0
name: listings
position:
  x: 375.650751621164
  y: 278.98403045165486
description:
  text: Load all data from a specific table.
  hash: 1a83f290
previewCodeHash: 0abff01b28784dc2
previewMode: "1000"
config:
  table_source:
    tableName: workspace.airbnb_source.listings
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "workspace.airbnb_source.listings"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_0.data"] = out["data"]
if globals().get("ld_display_outputs", False):
    display(ctx["source_0.data"])

In [0]:
"""
id: source_2
template: source
templateVersion: 2.0.0
name: hosts
position:
  x: 375.650751621164
  y: 568.9840304516548
description:
  text: Read CSV file from specified path.
  hash: 515d0985
previewCodeHash: fc2fe0e0a3e089ca
previewMode: "1000"
config:
  file_source:
    path: /Volumes/workspace/airbnb_source/hosts
    format: '"csv"'
    ignoreLeadingWhiteSpace: true
    ignoreTrailingWhiteSpace: true
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "file_source": {
        "path": "/Volumes/workspace/airbnb_source/hosts",
        "format": "\"csv\"",
        "ignoreLeadingWhiteSpace": True,
        "ignoreTrailingWhiteSpace": True
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_2.data"] = out["data"]
if globals().get("ld_display_outputs", False):
    display(ctx["source_2.data"])

In [0]:
"""
id: python_5
template: python
templateVersion: 1.0.0
name: bookings_cl
position:
  x: 696.7104715186782
  y: 429.52620341436415
description:
  text: Remove duplicate rows and fill missing values in booking details with defaults.
  hash: 6280618f
previewCodeHash: 18668eadd502268c
previewMode: "1000"
config:
  code: |
    if inputs.get("data"):
        df = inputs["data"][0]
        df = df.dropDuplicates()
        df = df.fillna(0,['nights_booked','booking_amount','cleaning_fee','service_fee'])
        df = df.fillna('Unknown',['booking_status'])
        result = df
    else:
        result = spark.createDataFrame([], "col: string")
input:
  - node: python_1
    input_port: data
    output_port: result
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        df = inputs["data"][0]
        df = df.dropDuplicates()
        df = df.fillna(0,['nights_booked','booking_amount','cleaning_fee','service_fee'])
        df = df.fillna('Unknown',['booking_status'])
        result = df
    else:
        result = spark.createDataFrame([], "col: string")

    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {
    "data": [
        ctx["python_1.result"]
    ],
    "data__sources": [
        {
            "node": "python_1",
            "output_port": "result",
            "name": "bookings_API",
            "df_name": "bookings_API"
        }
    ]
}
out = run(config, inputs, spark)
ctx["python_5.result"] = out["result"]
if globals().get("ld_display_outputs", False):
    display(ctx["python_5.result"])

In [0]:
"""
id: python_4
template: python
templateVersion: 1.0.0
name: listings_cl
position:
  x: 690.8104471046157
  y: 277.5324595178798
description:
  text: Remove duplicates and fill missing property details and numeric values with default placeholders.
  hash: f5208fe9
previewCodeHash: ab0e72914c0e273a
previewMode: "1000"
config:
  code: |
    import pandas as pd
    if inputs.get("data"):
        df = inputs["data"][0]
        df = df.dropDuplicates()
        df = df.fillna({"property_type": "Unknown", 
                    "room_type": "Unknown",
                     "city": "Unknown",
                     "country": "Unknown"})
        df = df.fillna(0,['accommodates','bedrooms','bathrooms','price_per_night'])
        result = df
    else:
        result = spark.createDataFrame([], "col: string")
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    import pandas as pd
    if inputs.get("data"):
        df = inputs["data"][0]
        df = df.dropDuplicates()
        df = df.fillna({"property_type": "Unknown", 
                    "room_type": "Unknown",
                     "city": "Unknown",
                     "country": "Unknown"})
        df = df.fillna(0,['accommodates','bedrooms','bathrooms','price_per_night'])
        result = df
    else:
        result = spark.createDataFrame([], "col: string")

    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {
    "data": [
        ctx["source_0.data"]
    ],
    "data__sources": [
        {
            "node": "source_0",
            "output_port": "data",
            "name": "listings",
            "df_name": "listings"
        }
    ]
}
out = run(config, inputs, spark)
ctx["python_4.result"] = out["result"]
if globals().get("ld_display_outputs", False):
    display(ctx["python_4.result"])

In [0]:
"""
id: python_6
template: python
templateVersion: 1.0.0
name: hosts_cl
position:
  x: 690.8104471046157
  y: 567.5324595178798
description:
  text: Remove duplicate records, drop a specific helper column, and fill missing host names and response rates with defaults.
  hash: 7eab377f
previewCodeHash: e038afa5d7b0d9ce
previewMode: "1000"
config:
  code: |
    from pyspark.sql.functions import col

    if inputs.get("data"):
        df = inputs["data"][0]
        df = df.dropDuplicates()
        df = df.drop(col('_rescued_data'))
        df = df.fillna({'host_name':'Name Missing',
            'response_rate':0 })
        result = df
    else:
        result = spark.createDataFrame([], "col: string")
input:
  - node: source_2
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    from pyspark.sql.functions import col

    if inputs.get("data"):
        df = inputs["data"][0]
        df = df.dropDuplicates()
        df = df.drop(col('_rescued_data'))
        df = df.fillna({'host_name':'Name Missing',
            'response_rate':0 })
        result = df
    else:
        result = spark.createDataFrame([], "col: string")

    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {
    "data": [
        ctx["source_2.data"]
    ],
    "data__sources": [
        {
            "node": "source_2",
            "output_port": "data",
            "name": "hosts",
            "df_name": "hosts"
        }
    ]
}
out = run(config, inputs, spark)
ctx["python_6.result"] = out["result"]
if globals().get("ld_display_outputs", False):
    display(ctx["python_6.result"])

In [0]:
"""
id: join_7
template: join
templateVersion: 2.0.0
name: listings_bookings_join
position:
  x: 1005.8847238090098
  y: 306.0390067087585
description:
  text: Keep all rows from the right table and only matching rows from the left table based on listing_id.
  hash: 5d8916fd
previewCodeHash: 5d9457a8cd0abaf6
previewMode: "1000"
config:
  join_type: right
  join_keys:
    - left: listing_id
      right: listing_id
  join_conditions: ""
  match_case: false
  left_columns:
    edits: []
    ordered: []
  right_columns:
    edits:
      - column: listing_id
        checked: false
      - column: created_at
        checked: false
    ordered: []
input:
  - node: python_4
    input_port: left
    output_port: result
  - node: python_5
    input_port: right
    output_port: result
"""

# generated from the system
from typing import Any, Dict, List
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(side: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + side + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(df_left, df_right, left_cfg: Dict[str, Any], right_cfg: Dict[str, Any]):
    left_cols = list(df_left.columns)
    right_cols = list(df_right.columns)
    left_edits = _edits_by_column(left_cfg)
    right_edits = _edits_by_column(right_cfg)
    left_lower = {c.lower() for c in left_cols}

    out = []
    for name in _ordered_names(left_cols, left_cfg):
        edit = left_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("left", name)
        alias = edit.get("alias") if edit else None
        out.append(col.alias(alias) if alias else col.alias(name))
    for name in _ordered_names(right_cols, right_cfg):
        edit = right_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("right", name)
        alias = edit.get("alias") if edit else None
        if alias:
            out.append(col.alias(alias))
        elif name.lower() in left_lower:
            out.append(col.alias("right_" + name))
        else:
            out.append(col.alias(name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_keys: List[Dict[str, str]] = config.get("join_keys", [])
    join_condition = config.get("join_conditions", "")
    join_type = config.get("join_type") or "split_join"
    match_case = config.get("match_case", False)
    left_cfg = config.get("left_columns") or {}
    right_cfg = config.get("right_columns") or {}

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    left_types = {f.name.lower(): f.dataType for f in df_left.schema}
    right_types = {f.name.lower(): f.dataType for f in df_right.schema}

    def key_col(side, name, types):
        col = _qualified_col(side, name)
        if not match_case and isinstance(types.get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    predicates = []
    for key in join_keys:
        predicates.append(
            key_col("left", key["left"], left_types)
            == key_col("right", key["right"], right_types)
        )
    if join_condition:
        predicates.append(F.expr(join_condition))

    join_expr = None
    for predicate in predicates:
        join_expr = predicate if join_expr is None else join_expr & predicate

    is_split = join_type == "split_join"
    matched_how = "inner" if is_split else join_type

    if join_expr is None:
        matched = df_left.join(df_right, how=matched_how)
    else:
        matched = df_left.join(df_right, join_expr, how=matched_how)

    projection = _projection(df_left, df_right, left_cfg, right_cfg)
    if projection:
        matched = matched.select(*projection)

    if is_split:
        if join_expr is None:
            left_unmatched = df_left.join(df_right, how="left_anti")
            right_unmatched = df_right.join(df_left, how="left_anti")
        else:
            left_unmatched = df_left.join(df_right, join_expr, how="left_anti")
            right_unmatched = df_right.join(df_left, join_expr, how="left_anti")
    else:
        left_unmatched = spark.createDataFrame([], df_left.schema)
        right_unmatched = spark.createDataFrame([], df_right.schema)

    return {
        "joined_data": matched,
        "left_unmatched": left_unmatched,
        "right_unmatched": right_unmatched,
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "right",
    "join_keys": [
        {
            "left": "listing_id",
            "right": "listing_id"
        }
    ],
    "join_conditions": "",
    "match_case": False,
    "left_columns": {
        "edits": [],
        "ordered": []
    },
    "right_columns": {
        "edits": [
            {
                "column": "listing_id",
                "checked": False
            },
            {
                "column": "created_at",
                "checked": False
            }
        ],
        "ordered": []
    }
}
inputs = {
    "left": ctx["python_4.result"],
    "right": ctx["python_5.result"]
}
out = run(config, inputs, spark)
ctx["join_7.joined_data"] = out["joined_data"]
ctx["join_7.left_unmatched"] = out["left_unmatched"]
ctx["join_7.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False):
    display(ctx["join_7.joined_data"])
    display(ctx["join_7.left_unmatched"])
    display(ctx["join_7.right_unmatched"])

In [0]:
"""
id: join_8
template: join
templateVersion: 2.0.0
name: OBT
position:
  x: 1264.05134467682
  y: 411.82549945011044
description:
  text: Join two sets of data on the host_id column from the left set only, excluding some columns from the right set.
  hash: 0e7e8c94
previewCodeHash: 1bd557a56bf9a475
previewMode: "1000"
config:
  join_type: left
  join_keys:
    - left: host_id
      right: host_id
  join_conditions: ""
  match_case: false
  left_columns:
    edits: []
    ordered: []
  right_columns:
    edits:
      - column: host_id
        checked: false
      - column: created_at
        checked: false
    ordered: []
input:
  - node: join_7
    input_port: left
    output_port: joined_data
  - node: python_6
    input_port: right
    output_port: result
"""

# generated from the system
from typing import Any, Dict, List
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(side: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + side + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(df_left, df_right, left_cfg: Dict[str, Any], right_cfg: Dict[str, Any]):
    left_cols = list(df_left.columns)
    right_cols = list(df_right.columns)
    left_edits = _edits_by_column(left_cfg)
    right_edits = _edits_by_column(right_cfg)
    left_lower = {c.lower() for c in left_cols}

    out = []
    for name in _ordered_names(left_cols, left_cfg):
        edit = left_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("left", name)
        alias = edit.get("alias") if edit else None
        out.append(col.alias(alias) if alias else col.alias(name))
    for name in _ordered_names(right_cols, right_cfg):
        edit = right_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("right", name)
        alias = edit.get("alias") if edit else None
        if alias:
            out.append(col.alias(alias))
        elif name.lower() in left_lower:
            out.append(col.alias("right_" + name))
        else:
            out.append(col.alias(name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_keys: List[Dict[str, str]] = config.get("join_keys", [])
    join_condition = config.get("join_conditions", "")
    join_type = config.get("join_type") or "split_join"
    match_case = config.get("match_case", False)
    left_cfg = config.get("left_columns") or {}
    right_cfg = config.get("right_columns") or {}

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    left_types = {f.name.lower(): f.dataType for f in df_left.schema}
    right_types = {f.name.lower(): f.dataType for f in df_right.schema}

    def key_col(side, name, types):
        col = _qualified_col(side, name)
        if not match_case and isinstance(types.get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    predicates = []
    for key in join_keys:
        predicates.append(
            key_col("left", key["left"], left_types)
            == key_col("right", key["right"], right_types)
        )
    if join_condition:
        predicates.append(F.expr(join_condition))

    join_expr = None
    for predicate in predicates:
        join_expr = predicate if join_expr is None else join_expr & predicate

    is_split = join_type == "split_join"
    matched_how = "inner" if is_split else join_type

    if join_expr is None:
        matched = df_left.join(df_right, how=matched_how)
    else:
        matched = df_left.join(df_right, join_expr, how=matched_how)

    projection = _projection(df_left, df_right, left_cfg, right_cfg)
    if projection:
        matched = matched.select(*projection)

    if is_split:
        if join_expr is None:
            left_unmatched = df_left.join(df_right, how="left_anti")
            right_unmatched = df_right.join(df_left, how="left_anti")
        else:
            left_unmatched = df_left.join(df_right, join_expr, how="left_anti")
            right_unmatched = df_right.join(df_left, join_expr, how="left_anti")
    else:
        left_unmatched = spark.createDataFrame([], df_left.schema)
        right_unmatched = spark.createDataFrame([], df_right.schema)

    return {
        "joined_data": matched,
        "left_unmatched": left_unmatched,
        "right_unmatched": right_unmatched,
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_keys": [
        {
            "left": "host_id",
            "right": "host_id"
        }
    ],
    "join_conditions": "",
    "match_case": False,
    "left_columns": {
        "edits": [],
        "ordered": []
    },
    "right_columns": {
        "edits": [
            {
                "column": "host_id",
                "checked": False
            },
            {
                "column": "created_at",
                "checked": False
            }
        ],
        "ordered": []
    }
}
inputs = {
    "left": ctx["join_7.joined_data"],
    "right": ctx["python_6.result"]
}
out = run(config, inputs, spark)
ctx["join_8.joined_data"] = out["joined_data"]
ctx["join_8.left_unmatched"] = out["left_unmatched"]
ctx["join_8.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False):
    display(ctx["join_8.joined_data"])
    display(ctx["join_8.left_unmatched"])
    display(ctx["join_8.right_unmatched"])

In [0]:
"""
id: sql_8
template: sql
templateVersion: 1.0.0
name: business_metrics
position:
  x: 1530.9896886990578
  y: 335.8006134275055
description:
  text: Add accommodation type, total amount, and response quality labels based on given conditions.
  hash: 9868df7f
previewCodeHash: 3eddcd35d0be825b
previewMode: "1000"
config:
  query: "SELECT *,\r

    CASE WHEN accommodates > 4 THEN \"Large\"\r

    WHEN accommodates > 2 THEN \"Medium\"\r

    ELSE \"Single\" END AS accommodation_type,\r

    (nights_booked*price_per_night) +\r

    cleaning_fee + service_fee\r

    AS total_amount,\r

    CASE WHEN response_rate > 90 THEN \"Very Good\"\r

    WHEN response_rate > 80 THEN \"Good\"\r

    WHEN response_rate > 60 THEN \"Average\"\r

    ELSE \"Poor\" END AS response_rate_quality\r

    FROM OBT;"
input:
  - node: join_8
    input_port: data
    output_port: joined_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT *,\r\nCASE WHEN accommodates > 4 THEN \"Large\"\r\nWHEN accommodates > 2 THEN \"Medium\"\r\nELSE \"Single\" END AS accommodation_type,\r\n(nights_booked*price_per_night) +\r\ncleaning_fee + service_fee\r\nAS total_amount,\r\nCASE WHEN response_rate > 90 THEN \"Very Good\"\r\nWHEN response_rate > 80 THEN \"Good\"\r\nWHEN response_rate > 60 THEN \"Average\"\r\nELSE \"Poor\" END AS response_rate_quality\r\nFROM OBT;"
}
inputs = {
    "data": [
        ctx["join_8.joined_data"]
    ],
    "data__sources": [
        {
            "node": "join_8",
            "output_port": "joined_data",
            "name": "OBT",
            "df_name": "OBT"
        }
    ]
}
out = run(config, inputs, spark)
ctx["sql_8.result"] = out["result"]
if globals().get("ld_display_outputs", False):
    display(ctx["sql_8.result"])

In [0]:
"""
id: sql_10
template: sql
templateVersion: 1.0.0
name: revenue_analytics
position:
  x: 1808.5591621567705
  y: 435.5938318568132
description:
  text: Summarize total revenue and average price per night by year, quarter, and month.
  hash: 3f0b6c74
previewCodeHash: 60fdd353b825386e
previewMode: "1000"
config:
  query: "SELECT \r

    year(booking_date) as year,\r

    quarter(booking_date) as quarter,\r

    month(booking_date) as month,\r

    sum(total_amount) as total_revenue,\r

    round(avg(price_per_night),2) as avg_price_per_night\r

    FROM business_metrics\r

    GROUP by \r

    year(booking_date),\r

    quarter(booking_date),\r

    month(booking_date);"
input:
  - node: sql_8
    input_port: data
    output_port: result
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT \r\nyear(booking_date) as year,\r\nquarter(booking_date) as quarter,\r\nmonth(booking_date) as month,\r\nsum(total_amount) as total_revenue,\r\nround(avg(price_per_night),2) as avg_price_per_night\r\nFROM business_metrics\r\nGROUP by \r\nyear(booking_date),\r\nquarter(booking_date),\r\nmonth(booking_date);"
}
inputs = {
    "data": [
        ctx["sql_8.result"]
    ],
    "data__sources": [
        {
            "node": "sql_8",
            "output_port": "result",
            "name": "business_metrics",
            "df_name": "business_metrics"
        }
    ]
}
out = run(config, inputs, spark)
ctx["sql_10.result"] = out["result"]
if globals().get("ld_display_outputs", False):
    display(ctx["sql_10.result"])

In [0]:
"""
id: sql_11
template: sql
templateVersion: 1.0.0
name: host_analytics
position:
  x: 1808.5591621567705
  y: 580.5938318568133
description:
  text: Calculate revenue and accommodation statistics by year, quarter, month, and host.
  hash: 287aef27
previewCodeHash: fcc2c794509dc47e
previewMode: "1000"
config:
  query: "SELECT \r

    year(booking_date) as year,\r

    quarter(booking_date) as quarter,\r

    month(booking_date) as month,\r

    host_name,\r

    sum(total_amount) as total_revenue,\r

    round(avg(price_per_night),2) as avg_price_per_night,\r

    sum(accommodates) as total_accommodates,\r

    sum(case when booking_status = 'cancelled' then 1 else 0 end) AS total_cancelled,\r

    sum(case when accommodation_type='Large' then 1 else 0 end)\r

    AS large_accommodates\r

    FROM business_metrics\r

    GROUP by \r

    year(booking_date),\r

    quarter(booking_date),\r

    month(booking_date),\r

    host_name"
input:
  - node: sql_8
    input_port: data
    output_port: result
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT \r\nyear(booking_date) as year,\r\nquarter(booking_date) as quarter,\r\nmonth(booking_date) as month,\r\nhost_name,\r\nsum(total_amount) as total_revenue,\r\nround(avg(price_per_night),2) as avg_price_per_night,\r\nsum(accommodates) as total_accommodates,\r\nsum(case when booking_status = 'cancelled' then 1 else 0 end) AS total_cancelled,\r\nsum(case when accommodation_type='Large' then 1 else 0 end)\r\nAS large_accommodates\r\nFROM business_metrics\r\nGROUP by \r\nyear(booking_date),\r\nquarter(booking_date),\r\nmonth(booking_date),\r\nhost_name"
}
inputs = {
    "data": [
        ctx["sql_8.result"]
    ],
    "data__sources": [
        {
            "node": "sql_8",
            "output_port": "result",
            "name": "business_metrics",
            "df_name": "business_metrics"
        }
    ]
}
out = run(config, inputs, spark)
ctx["sql_11.result"] = out["result"]
if globals().get("ld_display_outputs", False):
    display(ctx["sql_11.result"])

In [0]:
"""
id: sql_12
template: sql
templateVersion: 1.0.0
name: listing_analytics
position:
  x: 1808.5591621567705
  y: 725.5938318568133
description:
  text: Summarize bookings by year, quarter, month, and listing, showing total bookings, revenue, average accommodates, and cancellations.
  hash: 8648176e
previewCodeHash: 77e75b17b05220e0
previewMode: "1000"
config:
  query: "SELECT \r

    year(booking_date) as year,\r

    quarter(booking_date) as quarter,\r

    month(booking_date) as month,\r

    listing_id,\r

    count(distinct booking_id) as total_bookings,\r

    sum(total_amount) as total_revenue,\r

    avg(accommodates) avg_accommodates,\r

    sum(case when booking_status = 'cancelled' then 1 else 0 end) AS total_cancelled\r

    FROM business_metrics\r

    GROUP by \r

    year(booking_date),\r

    quarter(booking_date),\r

    month(booking_date),\r

    listing_id"
input:
  - node: sql_8
    input_port: data
    output_port: result
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT \r\nyear(booking_date) as year,\r\nquarter(booking_date) as quarter,\r\nmonth(booking_date) as month,\r\nlisting_id,\r\ncount(distinct booking_id) as total_bookings,\r\nsum(total_amount) as total_revenue,\r\navg(accommodates) avg_accommodates,\r\nsum(case when booking_status = 'cancelled' then 1 else 0 end) AS total_cancelled\r\nFROM business_metrics\r\nGROUP by \r\nyear(booking_date),\r\nquarter(booking_date),\r\nmonth(booking_date),\r\nlisting_id"
}
inputs = {
    "data": [
        ctx["sql_8.result"]
    ],
    "data__sources": [
        {
            "node": "sql_8",
            "output_port": "result",
            "name": "business_metrics",
            "df_name": "business_metrics"
        }
    ]
}
out = run(config, inputs, spark)
ctx["sql_12.result"] = out["result"]
if globals().get("ld_display_outputs", False):
    display(ctx["sql_12.result"])

In [0]:
"""
id: sql_9
template: sql
templateVersion: 1.0.0
name: booking_analytics
position:
  x: 1804.6151881522342
  y: 212.10197159975633
description:
  text: Calculate booking statistics, including total bookings, hosts, listings, accommodates, cancellations, and cancellation rate by year, quarter, and month.
  hash: 9b7a99fc
previewCodeHash: a1e1aba51952aca3
previewMode: full
config:
  query: "With cte as (SELECT \r

    year(booking_date) as year,\r

    quarter(booking_date) as quarter,\r

    month(booking_date) as month,\r

    count(distinct booking_id) as total_bookings,\r

    count(distinct host_id) as total_hosts,\r

    count(distinct listing_id) as total_listing,\r

    sum(accommodates) as total_accommodates,\r

    sum(case when booking_status = 'cancelled' then 1 else 0 end) AS total_cancelled\r

    FROM business_metrics\r

    GROUP by \r

    year(booking_date),\r

    quarter(booking_date),\r

    month(booking_date))\r

    select *, \r

    round((total_cancelled/total_bookings),2) as cancellation_rate\r

    from cte"
input:
  - node: sql_8
    input_port: data
    output_port: result
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "query": "With cte as (SELECT \r\nyear(booking_date) as year,\r\nquarter(booking_date) as quarter,\r\nmonth(booking_date) as month,\r\ncount(distinct booking_id) as total_bookings,\r\ncount(distinct host_id) as total_hosts,\r\ncount(distinct listing_id) as total_listing,\r\nsum(accommodates) as total_accommodates,\r\nsum(case when booking_status = 'cancelled' then 1 else 0 end) AS total_cancelled\r\nFROM business_metrics\r\nGROUP by \r\nyear(booking_date),\r\nquarter(booking_date),\r\nmonth(booking_date))\r\nselect *, \r\nround((total_cancelled/total_bookings),2) as cancellation_rate\r\nfrom cte"
}
inputs = {
    "data": [
        ctx["sql_8.result"]
    ],
    "data__sources": [
        {
            "node": "sql_8",
            "output_port": "result",
            "name": "business_metrics",
            "df_name": "business_metrics"
        }
    ]
}
out = run(config, inputs, spark)
ctx["sql_9.result"] = out["result"]
if globals().get("ld_display_outputs", False):
    display(ctx["sql_9.result"])

In [0]:
"""
id: output_14
template: output
templateVersion: 3.0.0
name: lakeflow_designer.enriched.revenue_analytics
position:
  x: 2068.5591621567705
  y: 435.5938318568132
description:
  text: Overwrite or create the specified table with new data.
  hash: "81768479"
previewMode: "1000"
config:
  output_type: table
  catalog: lakeflow_designer
  schema: enriched
  table_name: revenue_analytics
  write_mode: overwrite
input:
  - node: sql_10
    input_port: data
    output_port: result
"""

# generated from the system
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_EXCEL_CELLS = 5_000_000

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _file_path(catalog: str, schema: str, volume: str, file_name: str) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        schema_segment = schema.replace(".", "/")
        return f"/Volumes/{catalog}/{schema_segment}/{volume}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _write_single_file(rows, columns, file_type: str, path: str, append: bool) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                else:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        import io

        try:
            import openpyxl
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "Writing Excel files requires the 'openpyxl' package, which "
                "is not installed in this environment. Add "
                "openpyxl==3.1.5 to the notebook environment "
                "and apply it, then run again."
            ) from exc

        import pandas as pd
        import re as _re

        try:
            from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
        except ImportError:
            _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

        def _excel_safe(value):
            if isinstance(value, str):
                return _lb_illegal.sub("", value)
            if isinstance(value, (int, float, bool, type(None))):
                return value
            return _lb_illegal.sub("", str(value))

        header = list(columns)
        source_keys = list(columns)
        existing_values: List = []
        if appending:
            workbook = openpyxl.load_workbook(path, read_only=True)
            try:
                sheet = workbook.active
                row_iter = sheet.iter_rows(values_only=True)
                first = next(row_iter, None)
                if first is not None:
                    header = list(first)
                    if sorted(str(c) for c in header) != sorted(
                        str(_excel_safe(c)) for c in columns
                    ):
                        raise ValueError(
                            f"Output: cannot append to {path}: the data's columns "
                            f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                            f"file's columns {[str(c) for c in header]}."
                        )
                    raw_by_safe: Dict[str, Any] = {}
                    for c in columns:
                        raw_by_safe.setdefault(str(_excel_safe(c)), c)
                    source_keys = [raw_by_safe[str(c)] for c in header]
                    row_cap = _excel_row_cap(len(header))
                    for record in row_iter:
                        if len(existing_values) + len(rows) >= row_cap:
                            raise ValueError(
                                f"Output: appending {len(rows):,} rows would grow {path} past "
                                f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                                f"at {len(header):,} columns) — write to a table instead."
                            )
                        existing_values.append(list(record))
            finally:
                workbook.close()

        combined = existing_values + [
            [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
        ]
        new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])

        def _write_excel(tmp_path: str) -> None:
            buffer = io.BytesIO()
            new_df.to_excel(buffer, index=False, engine="openpyxl")
            with open(tmp_path, "wb") as out:
                out.write(buffer.getvalue())

        _stage(_write_excel)
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        _with_extension(config.get("file_name", ""), file_type),
    )
    append = config.get("write_mode", "overwrite") == "append"
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
    else:
        rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
        if len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
    _write_single_file(rows, df.columns, file_type, path, append)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
) -> None:
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        return {}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _run_materialized_view(config, source_view, spark)
        finally:
            spark.catalog.dropTempView(source_view)
        return {}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(stmt, args=args) if args else spark.sql(stmt)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "lakeflow_designer",
    "schema": "enriched",
    "table_name": "revenue_analytics",
    "write_mode": "overwrite"
}
inputs = {
    "data": ctx["sql_10.result"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: output_15
template: output
templateVersion: 3.0.0
name: lakeflow_designer.enriched.host_analytics
position:
  x: 2068.5591621567705
  y: 580.5938318568133
description:
  text: Write data to the table 'host_analytics' overwriting existing data.
  hash: 0d550697
previewMode: "1000"
config:
  output_type: table
  catalog: lakeflow_designer
  schema: enriched
  table_name: host_analytics
  write_mode: overwrite
input:
  - node: sql_11
    input_port: data
    output_port: result
"""

# generated from the system
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_EXCEL_CELLS = 5_000_000

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _file_path(catalog: str, schema: str, volume: str, file_name: str) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        schema_segment = schema.replace(".", "/")
        return f"/Volumes/{catalog}/{schema_segment}/{volume}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _write_single_file(rows, columns, file_type: str, path: str, append: bool) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                else:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        import io

        try:
            import openpyxl
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "Writing Excel files requires the 'openpyxl' package, which "
                "is not installed in this environment. Add "
                "openpyxl==3.1.5 to the notebook environment "
                "and apply it, then run again."
            ) from exc

        import pandas as pd
        import re as _re

        try:
            from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
        except ImportError:
            _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

        def _excel_safe(value):
            if isinstance(value, str):
                return _lb_illegal.sub("", value)
            if isinstance(value, (int, float, bool, type(None))):
                return value
            return _lb_illegal.sub("", str(value))

        header = list(columns)
        source_keys = list(columns)
        existing_values: List = []
        if appending:
            workbook = openpyxl.load_workbook(path, read_only=True)
            try:
                sheet = workbook.active
                row_iter = sheet.iter_rows(values_only=True)
                first = next(row_iter, None)
                if first is not None:
                    header = list(first)
                    if sorted(str(c) for c in header) != sorted(
                        str(_excel_safe(c)) for c in columns
                    ):
                        raise ValueError(
                            f"Output: cannot append to {path}: the data's columns "
                            f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                            f"file's columns {[str(c) for c in header]}."
                        )
                    raw_by_safe: Dict[str, Any] = {}
                    for c in columns:
                        raw_by_safe.setdefault(str(_excel_safe(c)), c)
                    source_keys = [raw_by_safe[str(c)] for c in header]
                    row_cap = _excel_row_cap(len(header))
                    for record in row_iter:
                        if len(existing_values) + len(rows) >= row_cap:
                            raise ValueError(
                                f"Output: appending {len(rows):,} rows would grow {path} past "
                                f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                                f"at {len(header):,} columns) — write to a table instead."
                            )
                        existing_values.append(list(record))
            finally:
                workbook.close()

        combined = existing_values + [
            [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
        ]
        new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])

        def _write_excel(tmp_path: str) -> None:
            buffer = io.BytesIO()
            new_df.to_excel(buffer, index=False, engine="openpyxl")
            with open(tmp_path, "wb") as out:
                out.write(buffer.getvalue())

        _stage(_write_excel)
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        _with_extension(config.get("file_name", ""), file_type),
    )
    append = config.get("write_mode", "overwrite") == "append"
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
    else:
        rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
        if len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
    _write_single_file(rows, df.columns, file_type, path, append)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
) -> None:
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        return {}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _run_materialized_view(config, source_view, spark)
        finally:
            spark.catalog.dropTempView(source_view)
        return {}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(stmt, args=args) if args else spark.sql(stmt)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "lakeflow_designer",
    "schema": "enriched",
    "table_name": "host_analytics",
    "write_mode": "overwrite"
}
inputs = {
    "data": ctx["sql_11.result"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: output_16
template: output
templateVersion: 3.0.0
name: lakeflow_designer.enriched.listing_analytics
position:
  x: 2068.5591621567705
  y: 725.5938318568133
description:
  text: Overwrite data in table named table_17.
  hash: 16f0de7c
previewMode: "1000"
config:
  output_type: table
  catalog: lakeflow_designer
  schema: enriched
  table_name: listing_analytics
  write_mode: overwrite
input:
  - node: sql_12
    input_port: data
    output_port: result
"""

# generated from the system
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_EXCEL_CELLS = 5_000_000

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _file_path(catalog: str, schema: str, volume: str, file_name: str) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        schema_segment = schema.replace(".", "/")
        return f"/Volumes/{catalog}/{schema_segment}/{volume}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _write_single_file(rows, columns, file_type: str, path: str, append: bool) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                else:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        import io

        try:
            import openpyxl
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "Writing Excel files requires the 'openpyxl' package, which "
                "is not installed in this environment. Add "
                "openpyxl==3.1.5 to the notebook environment "
                "and apply it, then run again."
            ) from exc

        import pandas as pd
        import re as _re

        try:
            from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
        except ImportError:
            _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

        def _excel_safe(value):
            if isinstance(value, str):
                return _lb_illegal.sub("", value)
            if isinstance(value, (int, float, bool, type(None))):
                return value
            return _lb_illegal.sub("", str(value))

        header = list(columns)
        source_keys = list(columns)
        existing_values: List = []
        if appending:
            workbook = openpyxl.load_workbook(path, read_only=True)
            try:
                sheet = workbook.active
                row_iter = sheet.iter_rows(values_only=True)
                first = next(row_iter, None)
                if first is not None:
                    header = list(first)
                    if sorted(str(c) for c in header) != sorted(
                        str(_excel_safe(c)) for c in columns
                    ):
                        raise ValueError(
                            f"Output: cannot append to {path}: the data's columns "
                            f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                            f"file's columns {[str(c) for c in header]}."
                        )
                    raw_by_safe: Dict[str, Any] = {}
                    for c in columns:
                        raw_by_safe.setdefault(str(_excel_safe(c)), c)
                    source_keys = [raw_by_safe[str(c)] for c in header]
                    row_cap = _excel_row_cap(len(header))
                    for record in row_iter:
                        if len(existing_values) + len(rows) >= row_cap:
                            raise ValueError(
                                f"Output: appending {len(rows):,} rows would grow {path} past "
                                f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                                f"at {len(header):,} columns) — write to a table instead."
                            )
                        existing_values.append(list(record))
            finally:
                workbook.close()

        combined = existing_values + [
            [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
        ]
        new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])

        def _write_excel(tmp_path: str) -> None:
            buffer = io.BytesIO()
            new_df.to_excel(buffer, index=False, engine="openpyxl")
            with open(tmp_path, "wb") as out:
                out.write(buffer.getvalue())

        _stage(_write_excel)
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        _with_extension(config.get("file_name", ""), file_type),
    )
    append = config.get("write_mode", "overwrite") == "append"
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
    else:
        rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
        if len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
    _write_single_file(rows, df.columns, file_type, path, append)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
) -> None:
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        return {}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _run_materialized_view(config, source_view, spark)
        finally:
            spark.catalog.dropTempView(source_view)
        return {}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(stmt, args=args) if args else spark.sql(stmt)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "lakeflow_designer",
    "schema": "enriched",
    "table_name": "listing_analytics",
    "write_mode": "overwrite"
}
inputs = {
    "data": ctx["sql_12.result"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: output_13
template: output
templateVersion: 3.0.0
name: lakeflow_designer.enriched.booking_analytics
position:
  x: 2064.615188152234
  y: 212.10197159975633
description:
  text: Overwrite or create the table 'booking_analytics' in lakeflow_designer.enriched schema with the input data.
  hash: 4db13cc8
previewMode: "1000"
config:
  output_type: table
  catalog: lakeflow_designer
  schema: enriched
  table_name: booking_analytics
  write_mode: overwrite
input:
  - node: sql_9
    input_port: data
    output_port: result
"""

# generated from the system
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_EXCEL_CELLS = 5_000_000

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _file_path(catalog: str, schema: str, volume: str, file_name: str) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        schema_segment = schema.replace(".", "/")
        return f"/Volumes/{catalog}/{schema_segment}/{volume}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _write_single_file(rows, columns, file_type: str, path: str, append: bool) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                else:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        import io

        try:
            import openpyxl
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "Writing Excel files requires the 'openpyxl' package, which "
                "is not installed in this environment. Add "
                "openpyxl==3.1.5 to the notebook environment "
                "and apply it, then run again."
            ) from exc

        import pandas as pd
        import re as _re

        try:
            from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
        except ImportError:
            _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

        def _excel_safe(value):
            if isinstance(value, str):
                return _lb_illegal.sub("", value)
            if isinstance(value, (int, float, bool, type(None))):
                return value
            return _lb_illegal.sub("", str(value))

        header = list(columns)
        source_keys = list(columns)
        existing_values: List = []
        if appending:
            workbook = openpyxl.load_workbook(path, read_only=True)
            try:
                sheet = workbook.active
                row_iter = sheet.iter_rows(values_only=True)
                first = next(row_iter, None)
                if first is not None:
                    header = list(first)
                    if sorted(str(c) for c in header) != sorted(
                        str(_excel_safe(c)) for c in columns
                    ):
                        raise ValueError(
                            f"Output: cannot append to {path}: the data's columns "
                            f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                            f"file's columns {[str(c) for c in header]}."
                        )
                    raw_by_safe: Dict[str, Any] = {}
                    for c in columns:
                        raw_by_safe.setdefault(str(_excel_safe(c)), c)
                    source_keys = [raw_by_safe[str(c)] for c in header]
                    row_cap = _excel_row_cap(len(header))
                    for record in row_iter:
                        if len(existing_values) + len(rows) >= row_cap:
                            raise ValueError(
                                f"Output: appending {len(rows):,} rows would grow {path} past "
                                f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                                f"at {len(header):,} columns) — write to a table instead."
                            )
                        existing_values.append(list(record))
            finally:
                workbook.close()

        combined = existing_values + [
            [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
        ]
        new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])

        def _write_excel(tmp_path: str) -> None:
            buffer = io.BytesIO()
            new_df.to_excel(buffer, index=False, engine="openpyxl")
            with open(tmp_path, "wb") as out:
                out.write(buffer.getvalue())

        _stage(_write_excel)
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        _with_extension(config.get("file_name", ""), file_type),
    )
    append = config.get("write_mode", "overwrite") == "append"
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
    else:
        rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
        if len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
    _write_single_file(rows, df.columns, file_type, path, append)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
) -> None:
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        return {}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _run_materialized_view(config, source_view, spark)
        finally:
            spark.catalog.dropTempView(source_view)
        return {}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(stmt, args=args) if args else spark.sql(stmt)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "lakeflow_designer",
    "schema": "enriched",
    "table_name": "booking_analytics",
    "write_mode": "overwrite"
}
inputs = {
    "data": ctx["sql_9.result"]
}
out = run(config, inputs, spark)